# Unit 04｜多轮主动学习循环

## Goal

用 GP + UCB 与 Random 在同一人工候选池运行八轮，并保存可审计 query log。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是粘合剂实验结果。


## Setup

策略函数只接收已标注 X/y、候选 X/ID 和 Oracle 回调；全局最优只在两条 campaign 完成后由 evaluator 读取。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

X_all = np.linspace(0.0, 12.0, 41).reshape(-1, 1)
oracle_values = (
    np.sin(0.9 * X_all[:, 0])
    + 0.35 * np.cos(2.2 * X_all[:, 0])
    + 0.06 * X_all[:, 0]
)
candidate_ids = np.array([f"C{i:02d}" for i in range(len(X_all))])
initial_indices = np.array([0, 13, 27, 40])

def offline_oracle(global_indices):
    indices = np.atleast_1d(global_indices).astype(int)
    values = oracle_values[indices].copy()
    return float(values[0]) if np.isscalar(global_indices) else values

initial_values = offline_oracle(initial_indices)
print("候选/初始:", len(X_all), len(initial_indices))


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 定义固定 GP

每轮建立同规格模型，避免上一轮对象中的隐藏状态。


In [ ]:
def make_gp():
    kernel = (
        ConstantKernel(1.0, constant_value_bounds="fixed")
        * RBF(1.0, length_scale_bounds="fixed")
        + WhiteKernel(0.02, noise_level_bounds="fixed")
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        optimizer=None,
        normalize_y=True,
    )

def select_with_tie_break(score, ids):
    order = np.lexsort((ids.astype(str), -np.asarray(score)))
    return int(order[0])


### 2. 实现策略循环

函数没有完整候选标签参数；局部位置映射到全局位置后，才调用 Oracle。


In [ ]:
def run_strategy(
    strategy,
    run_seed,
    initial_indices,
    initial_values,
    oracle,
    n_rounds=8,
    beta=1.2,
):
    labeled = list(np.asarray(initial_indices, dtype=int))
    labeled_y = list(np.asarray(initial_values, dtype=float))
    pool = [
        index for index in range(len(X_all))
        if index not in labeled
    ]
    rng = np.random.default_rng(run_seed)
    records = []

    for round_number in range(1, n_rounds + 1):
        model = make_gp()
        model.fit(X_all[labeled], np.asarray(labeled_y))
        mean, std = model.predict(X_all[pool], return_std=True)

        if strategy == "ucb":
            score = mean + beta * std
            query_local = select_with_tie_break(
                score,
                candidate_ids[pool],
            )
        elif strategy == "random":
            score = np.full(len(pool), np.nan)
            query_local = int(rng.integers(len(pool)))
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        query_global = int(pool[query_local])
        query_id = candidate_ids[query_global]
        prediction = float(mean[query_local])
        uncertainty = float(std[query_local])
        acquisition = (
            float(score[query_local])
            if strategy == "ucb"
            else np.nan
        )

        # 标签揭示线
        observed_y = float(oracle(query_global))
        labeled.append(query_global)
        labeled_y.append(observed_y)
        pool.remove(query_global)
        best_so_far = float(np.max(labeled_y))

        records.append({
            "strategy": strategy,
            "run_seed": run_seed,
            "round": round_number,
            "candidate_id": query_id,
            "prediction_mean": prediction,
            "prediction_std": uncertainty,
            "acquisition_score": acquisition,
            "observed_y": observed_y,
            "best_so_far": best_so_far,
            "n_labeled": len(labeled),
            "model_version": "fixed_gp_rbf_v1",
        })

    return pd.DataFrame(records)


### 3. 运行同初始集的两种策略

两条轨迹初始点和新增标签数相同；两条 campaign 完成后 evaluator 才读取全局最优。


In [ ]:
shared = {
    "initial_indices": initial_indices.copy(),
    "initial_values": initial_values.copy(),
    "oracle": offline_oracle,
}
ucb_log = run_strategy("ucb", run_seed=2026, **shared)
random_log = run_strategy("random", run_seed=2026, **shared)
query_log = pd.concat([ucb_log, random_log], ignore_index=True)

# 事后 evaluator：不再返回策略循环修改选点。
offline_global_best = float(oracle_values.max())
query_log["simple_regret"] = (
    offline_global_best - query_log["best_so_far"]
)
print(query_log.head(6).round(4).to_string(index=False))


### 4. 画 best-so-far 轨迹

单种子图只验证机制，不能证明 UCB 稳定优于 Random。


In [ ]:
for strategy, frame in query_log.groupby("strategy"):
    plt.plot(
        frame["round"],
        frame["best_so_far"],
        marker="o",
        label=strategy,
    )
plt.axhline(
    offline_global_best,
    color="black",
    linestyle="--",
    label="offline global best",
)
plt.xlabel("query round")
plt.ylabel("best observed value")
plt.title("Single-seed teaching trajectory")
plt.legend()
plt.show()


## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [ ]:
assert len(query_log) == 16
assert query_log.groupby("strategy")["candidate_id"].nunique().eq(8).all()
assert query_log.groupby("strategy")["n_labeled"].max().eq(12).all()
assert (query_log["simple_regret"] >= -1e-12).all()
assert query_log["model_version"].eq("fixed_gp_rbf_v1").all()
for _, frame in query_log.groupby("strategy"):
    assert frame["best_so_far"].is_monotonic_increasing
print("Unit 04 checks passed.")


## Next Steps

检查 query log 字段并完成索引练习。Unit 05 将把单种子轨迹扩展为公平多种子基准。
